# MVP — Framework de Análisis Predictivo para Sermukai

**Empresa:** Sermukai — outsourcing de servicios de seguridad, limpieza y jardinería en Jalisco, México.

**Objetivo:** Demostrar un flujo end-to-end de machine learning que transforme datos operativos desordenados en análisis predictivos confiables para la toma de decisiones de la junta directiva.

**Pregunta detonante:** ¿Cómo puede un framework modular de machine learning en Python transformar datos empresariales desordenados en análisis predictivos confiables para apoyar decisiones estratégicas?

In [ ]:
# Celda 1: Imports y configuración

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

# Semillas para reproducibilidad, sin esto cada ejecución podría marcar rondas distintas con anomalías, lo que haría difícil comparar resultados entre ejecuciones.
np.random.seed(42)

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")

print("Librerías cargadas correctamente.")

In [ ]:
# Celda 2:Carga de datos desde archivos CSV

def load_data(ruta="./"):
    """
    Carga los 5 archivos CSV en DataFrames y convierte columnas de fecha a datetime.

    Recibe: ruta del directorio donde están los CSVs.
    Devuelve: diccionario con DataFrames individuales.
    """
    df_empleados = pd.read_csv(f"{ruta}empleados.csv")
    df_locaciones = pd.read_csv(f"{ruta}locaciones.csv")
    df_turnos = pd.read_csv(f"{ruta}turnos.csv", parse_dates=["hora_inicio", "hora_fin"])
    df_rondas = pd.read_csv(f"{ruta}rondas.csv", parse_dates=["hora_inicio", "hora_final"])
    df_incidentes = pd.read_csv(f"{ruta}incidentes.csv", parse_dates=["fecha_hora"])

    dataframes = {
        "empleados": df_empleados,
        "locaciones": df_locaciones,
        "turnos": df_turnos,
        "rondas": df_rondas,
        "incidentes": df_incidentes,
    }

    print("Archivos CSV cargados:")
    for nombre, df in dataframes.items():
        print(f"  - {nombre}: {len(df)} registros, {len(df.columns)} columnas")

    return dataframes


# Los CSV viven en la carpeta data/ de la raíz del proyecto
dataframes = load_data("../data/")
dataframes["empleados"].head(3)

In [ ]:
# Celda 3 — JOINs entre tablas

def join_data(dataframes):
    """
    Realiza los JOINs necesarios entre las tablas para construir DataFrames consolidados.

    Recibe: diccionario con DataFrames individuales.
    Devuelve: DataFrames consolidados de turnos, rondas e incidentes.
    """
    df_empleados = dataframes["empleados"]
    df_locaciones = dataframes["locaciones"]
    df_turnos = dataframes["turnos"]
    df_rondas = dataframes["rondas"]
    df_incidentes = dataframes["incidentes"]

    # JOIN: turnos con empleados y locaciones
    df_turnos_completo = df_turnos.merge(
        df_empleados, left_on="empleado_id", right_on="id", suffixes=("", "_empleado")
    ).merge(
        df_locaciones, left_on="locacion_id", right_on="id", suffixes=("", "_locacion")
    )

    # JOIN: rondas con turnos completos
    df_rondas_completo = df_rondas.merge(
        df_turnos_completo, left_on="turno_id", right_on="id", suffixes=("", "_turno")
    )

    # JOIN: incidentes con turnos completos
    df_incidentes_completo = df_incidentes.merge(
        df_turnos_completo, left_on="turno_id", right_on="id", suffixes=("", "_turno")
    )

    print("JOINs realizados:")
    print(f"  - Turnos completos: {len(df_turnos_completo)} registros")
    print(f"  - Rondas completas: {len(df_rondas_completo)} registros")
    print(f"  - Incidentes completos: {len(df_incidentes_completo)} registros")

    return df_turnos_completo, df_rondas_completo, df_incidentes_completo


df_turnos_completo, df_rondas_completo, df_incidentes_completo = join_data(dataframes)
df_rondas_completo.head(3)

In [ ]:
# Celda 4 — Validación y limpieza de datos

def data_cleanup(df, nombre_df="DataFrame", columnas_nulos_permitidos=("categoria", "severidad")):
    """
    Detecta y elimina nulos y duplicados, y reporta los tipos de datos de un DataFrame.

    Recibe: DataFrame, su nombre descriptivo y las columnas cuyos nulos NO se eliminan
            (por defecto 'categoria' y 'severidad', que se rellenan en la Celda 7).
    Devuelve: DataFrame limpio y resumen de limpieza impreso.
    """
    print(f"\n{'='*50}")
    print(f"Limpieza de: {nombre_df} ({len(df)} registros)")
    print(f"{'='*50}")

    # Detectar nulos
    nulos = df.isnull().sum()
    nulos_total = nulos.sum()
    if nulos_total > 0:
        print(f"\nValores nulos encontrados ({nulos_total} total):")
        for col, count in nulos[nulos > 0].items():
            print(f"  - {col}: {count} nulos ({count/len(df)*100:.1f}%)")

        # Eliminar filas con nulos, salvo en las columnas que se rellenarán más adelante
        columnas_evaluar = [c for c in df.columns if c not in columnas_nulos_permitidos]
        antes = len(df)
        df = df.dropna(subset=columnas_evaluar)
        print(f"  → Filas con nulos eliminadas: {antes - len(df)}. Registros restantes: {len(df)}")
        print(f"  → Nulos conservados en: {[c for c in columnas_nulos_permitidos if c in df.columns]}")
    else:
        print("\nSin valores nulos.")

    # Detectar duplicados
    duplicados = df.duplicated().sum()
    if duplicados > 0:
        print(f"\nDuplicados encontrados: {duplicados}")
        df = df.drop_duplicates()
        print(f"  → Eliminados. Registros restantes: {len(df)}")
    else:
        print(f"Sin duplicados.")

    # Verificar tipos de datos
    print(f"\nTipos de datos:")
    for col, dtype in df.dtypes.items():
        print(f"  - {col}: {dtype}")

    return df


# Ejecutar limpieza en los DataFrames principales
df_rondas_completo = data_cleanup(df_rondas_completo, "Rondas (con JOIN)")
df_incidentes_completo = data_cleanup(df_incidentes_completo, "Incidentes (con JOIN)")

In [ ]:
# Celda 5 — Preprocesamiento

def pre_processing(df_rondas, df_incidentes):
    """
    Enriquece los DataFrames con columnas calculadas para el análisis.

    Recibe: DataFrames de rondas e incidentes (con JOINs).
    Devuelve: DataFrames enriquecidos con duración real, desviación, día de semana y hora.
    """
    # --- Rondas: calcular duración real y desviación ---
    df_rondas = df_rondas.copy()
    df_rondas["duracion_real_min"] = (
        (df_rondas["hora_final"] - df_rondas["hora_inicio"]).dt.total_seconds() / 60
    )
    df_rondas["desviacion_min"] = df_rondas["duracion_real_min"] - df_rondas["duracion_esperada_min"]

    print("Rondas preprocesadas:")
    print(f"  - Duración real promedio: {df_rondas['duracion_real_min'].mean():.1f} min")
    print(f"  - Desviación promedio: {df_rondas['desviacion_min'].mean():.1f} min")
    print(f"  - Desviación máxima: {df_rondas['desviacion_min'].max():.1f} min")

    # --- Incidentes: extraer día de semana y hora ---
    df_incidentes = df_incidentes.copy()
    dias_semana = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves",
                   4: "Viernes", 5: "Sábado", 6: "Domingo"}
    df_incidentes["dia_semana"] = df_incidentes["fecha_hora"].dt.weekday.map(dias_semana)
    df_incidentes["hora"] = df_incidentes["fecha_hora"].dt.hour

    # Limpieza básica de notas
    df_incidentes["notas_limpias"] = df_incidentes["notas"].str.lower().str.strip()

    print(f"\nIncidentes preprocesados:")
    print(f"  - Distribución por día: {df_incidentes['dia_semana'].value_counts().to_dict()}")

    return df_rondas, df_incidentes


# Ejecutar preprocesamiento
df_rondas_proc, df_incidentes_proc = pre_processing(df_rondas_completo, df_incidentes_completo)

In [ ]:
# Celda 6 — Detección de anomalías con IsolationForest

def detect_anomalies(df_rondas):
    """
    Usa IsolationForest para detectar rondas con desviaciones anómalas de duración.

    Recibe: DataFrame de rondas con columna 'desviacion_min'.
    Devuelve: DataFrame con columna 'es_anomalia' (True/False).
    """
    df_rondas = df_rondas.copy()

    modelo = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
    predicciones = modelo.fit_predict(df_rondas[["desviacion_min"]])

    # IsolationForest: -1 = anomalía, 1 = normal
    df_rondas["es_anomalia"] = predicciones == -1

    total = len(df_rondas)
    anomalas = df_rondas["es_anomalia"].sum()
    print(f"Detección de anomalías completada:")
    print(f"  - Total de rondas: {total}")
    print(f"  - Rondas anómalas: {anomalas} ({anomalas/total*100:.1f}%)")
    print(f"  - Rondas normales: {total - anomalas} ({(total-anomalas)/total*100:.1f}%)")
    print(f"\nDesviación promedio en rondas anómalas: "
          f"{df_rondas.loc[df_rondas['es_anomalia'], 'desviacion_min'].mean():.1f} min")

    return df_rondas


# Ejecutar detección de anomalías
df_rondas_proc = detect_anomalies(df_rondas_proc)

In [ ]:
# Celda 7 — Clasificación básica de incidentes por keywords

def classify_incidents(df_incidentes):
    """
    Clasifica incidentes usando reglas basadas en keywords del campo 'notas'.
    Rellena categoría y severidad donde sean nulos.

    Recibe: DataFrame de incidentes con columna 'notas_limpias'.
    Devuelve: DataFrame con 'categoria' y 'severidad' completados.
    """
    df = df_incidentes.copy()

    # Reglas de clasificación: keyword → (categoría, severidad)
    reglas = [
        (["alarma", "intrusión", "intrusion"], "Alarma activada", 4),
        (["sospechosa", "sospechoso", "merodeando"], "Persona sospechosa", 3),
        (["ruido", "extraño", "extraños"], "Actividad sospechosa", 2),
        (["puerta", "abierta", "emergencia"], "Acceso no autorizado", 3),
        (["vehículo", "vehiculo", "estacionado"], "Persona sospechosa", 2),
        (["cámara", "camara", "falla", "sistema"], "Falla eléctrica", 2),
        (["identificación", "identificacion", "ingresar", "sin identificación"], "Acceso no autorizado", 3),
        (["fuga", "agua", "rociadores"], "Daño a propiedad", 2),
        (["altercado", "verbal", "conflicto"], "Altercado", 4),
        (["ronda retrasada", "bloqueo", "retraso"], "Falla eléctrica", 1),
        (["humo", "incendio", "fuego"], "Incendio", 5),
        (["cortocircuito", "eléctrico", "tablero"], "Falla eléctrica", 4),
        (["grafiti", "grafitis", "muro"], "Vandalismo", 2),
        (["vidrio", "roto", "ventana"], "Daño a propiedad", 3),
        (["mareo", "médico", "apoyo médico"], "Emergencia médica", 3),
        (["pérdida", "equipo", "robo"], "Robo", 4),
        (["movimiento inusual", "movimiento"], "Intrusión perimetral", 4),
        (["animal", "silvestre"], "Otro", 1),
        (["evacuación", "evacuacion"], "Incendio", 4),
    ]

    def clasificar_nota(row):
        if pd.notna(row["categoria"]) and pd.notna(row["severidad"]):
            return row["categoria"], row["severidad"]

        nota = row["notas_limpias"] if pd.notna(row.get("notas_limpias")) else ""
        for keywords, cat, sev in reglas:
            if any(kw in nota for kw in keywords):
                return cat, sev
        return "Sin clasificar", 1

    resultados = df.apply(clasificar_nota, axis=1)
    df["categoria"] = resultados.apply(lambda x: x[0])
    df["severidad"] = resultados.apply(lambda x: x[1])

    nulos_antes = df_incidentes["categoria"].isnull().sum()
    print(f"Clasificación de incidentes completada:")
    print(f"  - Incidentes sin categoría (antes): {nulos_antes}")
    print(f"  - Incidentes sin categoría (después): {df['categoria'].isnull().sum()}")
    print(f"\nDistribución de categorías:")
    print(df["categoria"].value_counts().to_string())

    return df


# Ejecutar clasificación
df_incidentes_proc = classify_incidents(df_incidentes_proc)

In [ ]:
# Celda 8 — Evaluación de resultados

def evaluate_results(df_rondas, df_incidentes):
    """
    Genera un resumen estadístico de los resultados del análisis.

    Recibe: DataFrames de rondas (con anomalías) e incidentes (clasificados).
    Devuelve: Nada (imprime métricas).
    """
    print("=" * 60)
    print("RESUMEN DE RESULTADOS DEL ANÁLISIS")
    print("=" * 60)

    # --- Anomalías en rondas ---
    total_rondas = len(df_rondas)
    anomalas = df_rondas["es_anomalia"].sum()
    print(f"\n📊 ANOMALÍAS EN RONDAS:")
    print(f"  Total de rondas analizadas: {total_rondas}")
    print(f"  Rondas anómalas detectadas: {anomalas} ({anomalas/total_rondas*100:.1f}%)")
    print(f"  Desviación promedio (normales): "
          f"{df_rondas.loc[~df_rondas['es_anomalia'], 'desviacion_min'].mean():.1f} min")
    print(f"  Desviación promedio (anómalas): "
          f"{df_rondas.loc[df_rondas['es_anomalia'], 'desviacion_min'].mean():.1f} min")

    # --- Distribución de categorías ---
    print(f"\n📊 DISTRIBUCIÓN DE CATEGORÍAS DE INCIDENTES:")
    categorias = df_incidentes["categoria"].value_counts()
    for cat, count in categorias.items():
        print(f"  - {cat}: {count} ({count/len(df_incidentes)*100:.1f}%)")

    # --- Distribución de severidad ---
    print(f"\n📊 DISTRIBUCIÓN DE SEVERIDAD:")
    severidades = df_incidentes["severidad"].value_counts().sort_index()
    etiquetas_sev = {1: "Muy baja", 2: "Baja", 3: "Media", 4: "Alta", 5: "Crítica"}
    for sev, count in severidades.items():
        etiqueta = etiquetas_sev.get(int(sev), f"Nivel {sev}")
        print(f"  - Severidad {int(sev)} ({etiqueta}): {count} incidentes")

    # --- Incidentes por locación ---
    if "nombre_locacion" in df_incidentes.columns:
        col_loc = "nombre_locacion"
    elif "nombre_turno" in df_incidentes.columns:
        col_loc = "nombre_turno"
    else:
        col_loc = None

    if col_loc:
        print(f"\n📊 INCIDENTES POR LOCACIÓN (top 10):")
        por_locacion = df_incidentes[col_loc].value_counts().head(10)
        for loc, count in por_locacion.items():
            print(f"  - {loc}: {count}")


# Ejecutar evaluación
evaluate_results(df_rondas_proc, df_incidentes_proc)

In [ ]:
# Celda 9 — Heatmap: Concentración de incidentes por día de la semana vs hora

def present_heatmap(df_incidentes):
    """
    Genera un heatmap que muestra la concentración de incidentes por día y hora.

    Recibe: DataFrame de incidentes con columnas 'dia_semana' y 'hora'.
    Devuelve: Nada (renderiza la gráfica).
    """
    orden_dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

    pivot = df_incidentes.pivot_table(
        index="dia_semana",
        columns="hora",
        values="id",
        aggfunc="count",
        fill_value=0
    )

    # Reindexar para asegurar orden correcto y todas las horas
    pivot = pivot.reindex(index=orden_dias, fill_value=0)
    todas_horas = list(range(24))
    pivot = pivot.reindex(columns=todas_horas, fill_value=0)

    fig, ax = plt.subplots(figsize=(16, 6))
    sns.heatmap(
        pivot,
        cmap="YlOrRd",
        annot=True,
        fmt="d",
        linewidths=0.5,
        cbar_kws={"label": "Cantidad de incidentes"},
        ax=ax
    )

    ax.set_title("Concentración de Incidentes por Día de la Semana y Hora del Día",
                 fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Hora del día (0-23)", fontsize=12)
    ax.set_ylabel("Día de la semana", fontsize=12)

    plt.tight_layout()
    plt.show()


# Ejecutar visualización
present_heatmap(df_incidentes_proc)

In [ ]:
# Celda 10 — Boxplot: Desviación de duración de rondas por locación

def present_boxplot(df_rondas):
    """
    Genera un boxplot que muestra la desviación de duración de rondas por locación.
    Muestra las 10 locaciones con más rondas para mantener la legibilidad.

    Recibe: DataFrame de rondas con columnas 'desviacion_min', 'nombre_locacion' y 'es_anomalia'.
    Devuelve: Nada (renderiza la gráfica).
    """
    # Determinar la columna de nombre de locación
    if "nombre_locacion" in df_rondas.columns:
        col_loc = "nombre_locacion"
    else:
        col_loc = "nombre_turno"

    # Filtrar a las 10 locaciones con más rondas
    top_locaciones = df_rondas[col_loc].value_counts().head(10).index
    df_filtrado = df_rondas[df_rondas[col_loc].isin(top_locaciones)]

    fig, ax = plt.subplots(figsize=(16, 7))

    sns.boxplot(
        data=df_filtrado,
        x=col_loc,
        y="desviacion_min",
        palette="Set2",
        ax=ax,
        fliersize=4
    )

    # Superponer puntos de anomalías detectadas
    anomalas = df_filtrado[df_filtrado["es_anomalia"]]
    if len(anomalas) > 0:
        sns.stripplot(
            data=anomalas,
            x=col_loc,
            y="desviacion_min",
            color="red",
            size=8,
            marker="X",
            alpha=0.7,
            ax=ax,
            label="Anomalía (IsolationForest)"
        )

    ax.set_title("Desviación de Duración de Rondas por Locación (Top 10)\n(Real vs Esperada)",
                 fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Locación", fontsize=12)
    ax.set_ylabel("Desviación en minutos (real - esperada)", fontsize=12)
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5, label="Sin desviación")

    plt.xticks(rotation=30, ha="right")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


# Ejecutar visualización
present_boxplot(df_rondas_proc)

## Conclusiones del Análisis

### Hallazgos del Heatmap (Concentración de Incidentes)
- Se observa una **mayor concentración de incidentes en horarios nocturnos** (20:00 - 03:00), lo cual es consistente con la naturaleza de las operaciones de seguridad.
- Los días **viernes y sábado** muestran un incremento notable en la frecuencia de incidentes, lo que sugiere la necesidad de **refuerzo de personal** en estas franjas.
- Las horas diurnas presentan menor actividad de incidentes, lo que podría justificar una redistribución de recursos hacia turnos nocturnos.

### Hallazgos del Boxplot (Desviación de Rondas)
- Algunas locaciones presentan **cajas más anchas**, lo que indica mayor variabilidad en la duración de las rondas respecto a lo esperado.
- Los puntos marcados como **anomalías por IsolationForest** (marcados en rojo) corresponden a rondas con desviaciones significativas, tanto por exceso como por defecto.
- Las locaciones con nivel de riesgo alto merecen atención especial cuando presentan desviaciones frecuentes.

### Valor del Framework
- El uso de **IsolationForest** permite detectar automáticamente rondas atípicas sin necesidad de reglas manuales, lo cual escala mejor conforme crecen los datos.
- La **clasificación por keywords** demuestra que incluso un enfoque simple de PLN puede llenar vacíos en datos incompletos (categoría y severidad faltantes).
- El flujo modular (carga → limpieza → preprocesamiento → análisis → evaluación → visualización) facilita el mantenimiento y la extensión del sistema.

### Siguientes pasos
1. **Conectar con datos reales** de la base de datos de Sermukai.
2. **Entrenar un modelo de PLN** para clasificación más precisa de incidentes.
3. **Implementar alertas automáticas** cuando se detecten anomalías en tiempo real.
4. **Desarrollar un dashboard interactivo** para la junta directiva.